# Phase 2 — Data preparation

This notebook downloads the real Tiny ImageNet, COCO val2017, COCO annotations, and ImageNet sample-image archives. It validates the raw data, creates the approved fixed splits, generates JSONL manifests, and packages the manifests for saving. It does not train models.

Use the Python 3.11 Colab runtime selected for Phase 1. After the first requirements installation, restart the Colab session if pip replaced packages, then run the notebook again from the top.

In [1]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/pranavshrivastava1104/deep_learning_project.git"
GIT_BRANCH = "testing"
PROJECT_ROOT = Path("/content/deep_learning_project")
DATA_ROOT = Path("/content/data")
MANIFEST_ROOT = PROJECT_ROOT / "data" / "manifests"
SEED = 42

def run(command: list[str], *, cwd: Path | None = None) -> None:
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True, text=True)

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f"Python 3.11 is required, but this runtime uses {sys.version.split()[0]}."
    )
print(f"Repository: {REPOSITORY_URL}")
print(f"Branch: {GIT_BRANCH}")
print(f"Raw data: {DATA_ROOT}")
print(f"Manifests: {MANIFEST_ROOT}")

Repository: https://github.com/pranavshrivastava1104/deep_learning_project.git
Branch: testing
Raw data: /content/data
Manifests: /content/deep_learning_project/data/manifests


## 1. Clone or update the project repository

In [2]:
if (PROJECT_ROOT / ".git").is_dir():
    run(["git", "checkout", GIT_BRANCH], cwd=PROJECT_ROOT)
    run(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
    run(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=PROJECT_ROOT)
else:
    run(
        [
            "git",
            "clone",
            "--branch",
            GIT_BRANCH,
            "--single-branch",
            REPOSITORY_URL,
            str(PROJECT_ROOT),
        ]
    )
print(f"✅ Repository ready on branch: {GIT_BRANCH}")

+ git clone --branch testing --single-branch https://github.com/pranavshrivastava1104/deep_learning_project.git /content/deep_learning_project
✅ Repository ready on branch: testing


## 2. Install the locked environment

If this first installation replaces Pillow or another already-imported package, restart the Colab session and rerun from the first cell. On the second run the installation is normally a no-op.

In [3]:
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--require-hashes",
        "-r",
        "requirements/colab.txt",
    ],
    cwd=PROJECT_ROOT,
)
run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."],
    cwd=PROJECT_ROOT,
)
print("✅ Locked requirements and project package installed")

+ /usr/bin/python3 -m pip install --require-hashes -r requirements/colab.txt
+ /usr/bin/python3 -m pip install --no-deps -e .
✅ Locked requirements and project package installed


## 3. Configure paths and deterministic seed

In [4]:
import sys

project_root_text = str(PROJECT_ROOT)
if project_root_text not in sys.path:
    sys.path.insert(0, project_root_text)
if not (PROJECT_ROOT / "models" / "__init__.py").is_file():
    raise RuntimeError(f"Project checkout is incomplete: {PROJECT_ROOT}")

from models.common.reproducibility import set_global_seed

DATA_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
set_global_seed(SEED)
print(f"✅ Seed configured: {SEED}")

✅ Seed configured: 42


In [5]:
import importlib
import pipelines.data.downloader as downloader

importlib.invalidate_caches()
downloader = importlib.reload(downloader)

print("Reloaded file:", downloader.__file__)
print("COCO URL:", downloader.DATASETS["coco_annotations"].url)

assert downloader.DATASETS["coco_annotations"].url.startswith(
    "https://s3.amazonaws.com/"
)

Reloaded file: /content/deep_learning_project/pipelines/data/downloader.py
COCO URL: https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval2017.zip


## 4. Download the real datasets

In [6]:
from pipelines.data.downloader import download_datasets

DOWNLOAD_ORDER = (
    "test_images",
    "coco_annotations",
    "tiny_imagenet",
    "coco_val2017",
)
download_results = download_datasets(DOWNLOAD_ORDER, DATA_ROOT)
for result in download_results:
    status = "downloaded" if result.created else "already present"
    print(f"✅ {result.dataset}: {status} at {result.path}")

✅ test_images: downloaded at /content/data/imagenet-sample-images
✅ coco_annotations: downloaded at /content/data/annotations
✅ tiny_imagenet: downloaded at /content/data/tiny-imagenet-200
✅ coco_val2017: downloaded at /content/data/val2017


## 5. Inspect and validate raw data

In [7]:
from pipelines.data.coco import validate_coco
from pipelines.data.similarity import validate_similarity_source
from pipelines.data.tiny_imagenet import validate_tiny_imagenet

for path in sorted(DATA_ROOT.iterdir()):
    print(path)

TINY_ROOT = DATA_ROOT / "tiny-imagenet-200"
COCO_IMAGES_ROOT = DATA_ROOT / "val2017"
COCO_ANNOTATIONS = DATA_ROOT / "annotations" / "instances_val2017.json"
SIMILARITY_SOURCE_ROOT = DATA_ROOT / "imagenet-sample-images"

tiny_raw = validate_tiny_imagenet(TINY_ROOT)
coco_raw = validate_coco(COCO_IMAGES_ROOT, COCO_ANNOTATIONS)
similarity_raw = validate_similarity_source(SIMILARITY_SOURCE_ROOT)

print(
    f"✅ Tiny ImageNet: classes={tiny_raw.class_count}, "
    f"train_images={tiny_raw.training_image_count}, "
    f"official_validation_images={tiny_raw.validation_image_count}"
)
print(
    f"✅ COCO: images={coco_raw.image_count}, "
    f"annotations={coco_raw.annotation_count}, "
    f"categories={coco_raw.category_count}"
)
print(f"✅ Similarity source images: {similarity_raw.image_count}")

/content/data/annotations
/content/data/imagenet-sample-images
/content/data/tiny-imagenet-200
/content/data/val2017
✅ Tiny ImageNet: classes=200, train_images=100000, official_validation_images=10000
✅ COCO: images=5000, annotations=36781, categories=80
✅ Similarity source images: 1000


## 6. Generate Tiny ImageNet manifests

Official training images are split per class into 90% training, 5% validation, and 5% calibration. The complete labelled official validation set becomes test.

In [8]:
from pipelines.data.tiny_imagenet import prepare_tiny_imagenet_manifests

tiny_manifests = prepare_tiny_imagenet_manifests(
    TINY_ROOT,
    DATA_ROOT,
    MANIFEST_ROOT,
    seed=SEED,
    validation_summary=tiny_raw,
)
print(tiny_manifests)
print("✅ Tiny ImageNet manifests generated")

TinyImageNetManifestSummary(output_directory=PosixPath('/content/deep_learning_project/data/manifests/tiny-imagenet/v1'), train_count=90000, validation_count=5000, calibration_count=5000, test_count=10000)
✅ Tiny ImageNet manifests generated


## 7. Generate COCO manifests

The 5,000 val2017 image IDs are deterministically partitioned into 500 calibration, 1,000 validation, and 3,500 test images.

In [9]:
from pipelines.data.coco import prepare_coco_manifests

coco_manifests = prepare_coco_manifests(
    COCO_IMAGES_ROOT,
    COCO_ANNOTATIONS,
    DATA_ROOT,
    MANIFEST_ROOT,
    seed=SEED,
    validation_summary=coco_raw,
)
print(coco_manifests)
print("✅ COCO manifests generated")

CocoManifestSummary(output_directory=PosixPath('/content/deep_learning_project/data/manifests/coco-2017/v1'), calibration_count=500, validation_count=1000, test_count=3500, category_count=80)
✅ COCO manifests generated


## 8. Generate similarity calibration, query, and gallery data

Ten percent of source images are reserved for calibration. Each remaining original becomes a gallery item, and a deterministic crop/resize/brightness variant becomes its positive query.

In [10]:
from pipelines.data.similarity import prepare_similarity_manifests

similarity_manifests = prepare_similarity_manifests(
    SIMILARITY_SOURCE_ROOT,
    DATA_ROOT,
    MANIFEST_ROOT,
    seed=SEED,
    validation_summary=similarity_raw,
)
print(similarity_manifests)
print("✅ Similarity manifests and generated images created")

SimilarityManifestSummary(output_directory=PosixPath('/content/deep_learning_project/data/manifests/similarity-gallery/v1'), processed_directory=PosixPath('/content/data/similarity-gallery/v1'), calibration_count=100, query_count=900, gallery_count=900, positive_pair_count=900)
✅ Similarity manifests and generated images created


## 9. Verify completion counts and save manifest outputs

In [11]:
import shutil

assert tiny_manifests.train_count == 90_000
assert tiny_manifests.validation_count == 5_000
assert tiny_manifests.calibration_count == 5_000
assert tiny_manifests.test_count == 10_000
assert coco_manifests.calibration_count == 500
assert coco_manifests.validation_count == 1_000
assert coco_manifests.test_count == 3_500
assert coco_manifests.category_count == 80
assert similarity_manifests.query_count == similarity_manifests.gallery_count
assert similarity_manifests.query_count == similarity_manifests.positive_pair_count

manifest_files = sorted(MANIFEST_ROOT.rglob("*.json*"))
if len(manifest_files) != 13:
    raise RuntimeError(f"Expected 13 manifest files, found {len(manifest_files)}")
for path in manifest_files:
    print(path.relative_to(PROJECT_ROOT))

archive_path = shutil.make_archive(
    "/content/phase2-manifests",
    "zip",
    root_dir=MANIFEST_ROOT,
)
run(["git", "status", "--short", "data/manifests"], cwd=PROJECT_ROOT)
print(f"✅ Phase 2 manifests verified and packaged: {archive_path}")
print("✅ Phase 2 data preparation completed")

data/manifests/coco-2017/v1/calibration.jsonl
data/manifests/coco-2017/v1/categories.json
data/manifests/coco-2017/v1/test.jsonl
data/manifests/coco-2017/v1/validation.jsonl
data/manifests/similarity-gallery/v1/calibration.jsonl
data/manifests/similarity-gallery/v1/gallery.jsonl
data/manifests/similarity-gallery/v1/positive_pairs.jsonl
data/manifests/similarity-gallery/v1/test_queries.jsonl
data/manifests/tiny-imagenet/v1/calibration.jsonl
data/manifests/tiny-imagenet/v1/labels.json
data/manifests/tiny-imagenet/v1/test.jsonl
data/manifests/tiny-imagenet/v1/train.jsonl
data/manifests/tiny-imagenet/v1/validation.jsonl
+ git status --short data/manifests
✅ Phase 2 manifests verified and packaged: /content/phase2-manifests.zip
✅ Phase 2 data preparation completed


In [12]:
import subprocess
import pipelines.data.downloader as downloader

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()

print("Git commit:", commit)
print("Downloader file:", downloader.__file__)
print("COCO URL:", downloader.DATASETS["coco_annotations"].url)

Git commit: b9263d60885dbdaff1245d7f4d49e3e83e26204e
Downloader file: /content/deep_learning_project/pipelines/data/downloader.py
COCO URL: https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval2017.zip
